# spatial-ml — Fine-tune SpaceLLaVA on Construction Data

**Before running:** Change GPU runtime to H100

This notebook:
1. Loads your VQA dataset from HuggingFace
2. Fine-tunes `remyxai/SpaceLLaVA` with LoRA (two-stage)
3. Pushes the checkpoint back to HuggingFace

## 0. Config — fill these in

In [10]:
from google.colab import userdata
HF_TOKEN        = userdata.get("HF_TOKEN")
DATASET_REPO    = "elizqiu/spatial-ml"
MODEL_OUT_REPO  = "elizqiu/spatialvlm-construction"
BASE_MODEL      = "llava-hf/llava-1.5-7b-hf"

# Training hyperparams
STAGE1_EPOCHS   = 1
STAGE2_EPOCHS   = 1
BATCH_SIZE      = 8
LR              = 2e-4
LORA_R          = 16
MAX_SEQ_LEN     = 512


## 1. Install dependencies

In [11]:
!pip install -q -U transformers peft accelerate huggingface_hub Pillow datasets torchao
import IPython
IPython.Application.instance().kernel.do_shutdown(True)


## 2. Check GPU

In [12]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

CUDA available: True
GPU: NVIDIA A100-SXM4-80GB
VRAM: 85.1 GB


## 3. Log in to HuggingFace

In [13]:
from huggingface_hub import login, HfApi
login(token=HF_TOKEN)
print("Logged in as:", HfApi().whoami()["name"])

Logged in as: elizqiu


## 4. Load dataset from HuggingFace

In [48]:
import json
from pathlib import Path
from huggingface_hub import hf_hub_download

# Download the VQA pairs JSONL
vqa_path = hf_hub_download(
    repo_id=DATASET_REPO,
    filename="data/vqa_pairs.jsonl",
    repo_type="dataset",
    token=HF_TOKEN,
)

with open(vqa_path) as f:
    all_pairs = [json.loads(line) for line in f if line.strip()]

print(f"Loaded {len(all_pairs)} QA pairs")
print("Example:", json.dumps(all_pairs[0], indent=2))

Loaded 42674 QA pairs
Example: {
  "image_path": "data/frames/05_production_mp/05_production_mp_f000000.jpg",
  "question": "What is the average depth of the construction site scene?",
  "answer": "The average depth of the scene is approximately 12.49 metres.",
  "type": "quantitative_distance",
  "objects_referenced": []
}


## 5. Download frames from HuggingFace

This downloads the images referenced by the QA pairs.
Only downloads once — skips if already present.

In [15]:
import os
from huggingface_hub import snapshot_download

frames_local = Path("/content/data/frames")
if not frames_local.exists():
    print("Downloading frames (this takes a few minutes)...")
    snapshot_download(
        repo_id=DATASET_REPO,
        repo_type="dataset",
        allow_patterns="data/frames/**/*",
        local_dir="/content",
        token=HF_TOKEN,
    )
    print("Done.")
else:
    print("Frames already downloaded.")

# Remap image_path to local Colab paths
# Original paths look like: data/frames/05_production_mp/05_production_mp_f000000.jpg
remapped = []
for pair in all_pairs:
    p = pair.get("image_path", "")
    # Strip any absolute prefix and anchor to /content
    rel = p.replace("/Users/eq/Documents/GithubProjects/spatial-ml/", "")
    local_path = f"/content/{rel}"
    if os.path.exists(local_path):
        pair["image_path"] = local_path
        remapped.append(pair)

print(f"{len(remapped)} pairs have matching local images (out of {len(all_pairs)})")
all_pairs = remapped

Frames already downloaded.
42674 pairs have matching local images (out of 42674)


In [16]:
import os
# Check if frames dir exists and has files
frames_path = "/content/frames"
if os.path.exists(frames_path):
    subdirs = os.listdir(frames_path)
    print(f"Found {len(subdirs)} items in /content/frames: {subdirs[:5]}")
else:
    print("/content/frames does not exist")

# Also check what's in /content
print("\n/content contents:", os.listdir("/content"))

/content/frames does not exist

/content contents: ['.config', '.cache', 'data', 'sample_data']


## 6. Dataset class

In [31]:
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torch.nn.utils.rnn import pad_sequence
import torch

class SpatialVQADataset(Dataset):
    def __init__(self, samples, processor, max_length=MAX_SEQ_LEN):
        self.samples = samples
        self.processor = processor
        self.max_length = max_length
        self.image_token_id = model.config.image_token_index

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        image = Image.open(s["image_path"]).convert("RGB")
        messages = [{"role": "user", "content": [
            {"type": "image"},
            {"type": "text", "text": "Question: " + s["question"] + "\nAnswer: " + s["answer"]},
        ]}]
        prompt = processor.apply_chat_template(messages, tokenize=False)
        pixel_values = processor.image_processor(image, return_tensors="pt")["pixel_values"]
        text_enc = processor.tokenizer(prompt, return_tensors="pt", truncation=True, max_length=self.max_length)
        input_ids = text_enc["input_ids"].squeeze(0).clone()
        input_ids[input_ids == 529] = self.image_token_id
        labels = input_ids.clone()
        labels[labels == processor.tokenizer.pad_token_id] = -100
        return {
            "input_ids": input_ids,
            "attention_mask": text_enc["attention_mask"].squeeze(0),
            "pixel_values": pixel_values.squeeze(0),
            "labels": labels,
        }

def collate_fn(batch):
    input_ids = pad_sequence([b["input_ids"] for b in batch], batch_first=True, padding_value=processor.tokenizer.pad_token_id)
    attention_mask = pad_sequence([b["attention_mask"] for b in batch], batch_first=True, padding_value=0)
    labels = pad_sequence([b["labels"] for b in batch], batch_first=True, padding_value=-100)
    pixel_values = torch.stack([b["pixel_values"] for b in batch])
    return {"input_ids": input_ids, "attention_mask": attention_mask, "pixel_values": pixel_values, "labels": labels}


## 7. Load base model + apply LoRA

In [18]:
from transformers import AutoProcessor, AutoModelForImageTextToText
from peft import LoraConfig, get_peft_model, TaskType

device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Loading {BASE_MODEL}...")
processor = AutoProcessor.from_pretrained(BASE_MODEL, token=HF_TOKEN)
model = AutoModelForImageTextToText.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.bfloat16,
    token=HF_TOKEN,
).to(device)

print(f"Loaded as: {type(model).__name__}")
print(f"image_token_index: {model.config.image_token_index}")

lora_cfg = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=LORA_R,
    lora_alpha=LORA_R * 2,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()


Loading remyxai/SpaceLLaVA...


Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] LlavaForConditionalGeneration LOAD REPORT from: remyxai/SpaceLLaVA
Key                                                                                            | Status     |                                                                                                 
-----------------------------------------------------------------------------------------------+------------+-------------------------------------------------------------------------------------------------
model.layers.{0...39}.input_layernorm.weight                                                   | UNEXPECTED |                                                                                                 
model.vision_tower.vision_tower.vision_model.encoder.layers.{0...23}.self_attn.out_proj.weight | UNEXPECTED |                                                                                                 
model.layers.{0...39}.self_attn.v_proj.weight                                             

Loaded as: LlavaForConditionalGeneration
trainable params: 9,961,472 || all params: 7,072,864,256 || trainable%: 0.1408


In [19]:
# (no longer needed)


## 8. Training loop

In [20]:
# (no longer needed)


In [21]:
from transformers import get_cosine_schedule_with_warmup

def freeze_vision(model):
    for name, param in model.named_parameters():
        if "vision" in name or "image_encoder" in name:
            param.requires_grad = False

def unfreeze_all(model):
    for param in model.parameters():
        param.requires_grad = True

def train_stage(model, dataloader, optimizer, scheduler, epochs, label):
    model.train()
    for epoch in range(epochs):
        total_loss = 0.0
        for i, batch in enumerate(dataloader):
            batch = {k: v.to(device) for k, v in batch.items()}
            loss = model(**batch).loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            total_loss += loss.item()
            if i % 50 == 0:
                print(f"  [{label}] epoch {epoch+1}/{epochs}  step {i}/{len(dataloader)}  loss={loss.item():.4f}")
        print(f"[{label}] epoch {epoch+1} avg loss={total_loss/len(dataloader):.4f}")

## 9. Stage 1 — freeze vision encoder, train on all construction VQA pairs

In [51]:
# (dataset class defined in ## 6. Dataset class above)


Train: 38406  Val: 4268
image token 32000 count: 8


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/linear.py:134: UserWarning: gemm_and_bias error: CUBLAS_STATUS_EXECUTION_FAILED when calling cublasLtMatmul with transpose_mat1 1 transpose_mat2 0 m 4096 n 4616 k 1024 mat1_ld 1024 mat2_ld 1024 result_ld 4096 abType 14 cType 14 computeType 68 scaleType 0. Will attempt to recover by calling unfused cublas path. (Triggered internally at /pytorch/aten/src/ATen/cuda/CUDABlas.cpp:1765.)
  return F.linear(input, self.weight, self.bias)


AcceleratorError: CUDA error: device-side assert triggered
Search for `cudaErrorAssert' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [49]:
import random
from transformers import get_cosine_schedule_with_warmup

split = int(len(all_pairs) * 0.9)
train_pairs = all_pairs[:split]
val_pairs = all_pairs[split:]
random.shuffle(train_pairs)
print(f"Train: {len(train_pairs)}  Val: {len(val_pairs)}")

ds1 = SpatialVQADataset(train_pairs, processor)
dl1 = DataLoader(ds1, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, collate_fn=collate_fn)

freeze_vision(model)
opt1 = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LR)
sched1 = get_cosine_schedule_with_warmup(opt1, num_warmup_steps=100, num_training_steps=STAGE1_EPOCHS * len(dl1))
train_stage(model, dl1, opt1, sched1, STAGE1_EPOCHS, "stage1")


Train: 38406  Val: 4268
Train: 38406  Val: 4268


ValueError: Image features and image tokens do not match, tokens: 0, features: 18874368

## 10. Stage 2 — unfreeze vision encoder, short pass on construction-specific pairs only

Filters for safety-alert and quantitative pairs to emphasise construction-domain knowledge.

In [ ]:
construction_types = {"safety_alert", "quantitative_distance"}
construction_pairs = [p for p in train_pairs if p.get("type") in construction_types]
print(f"Construction-specific pairs for stage 2: {len(construction_pairs)}")

if construction_pairs:
    ds2 = SpatialVQADataset(construction_pairs, processor)
    dl2 = DataLoader(ds2, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)

    unfreeze_all(model)
    opt2 = torch.optim.AdamW(model.parameters(), lr=LR / 5)
    sched2 = get_cosine_schedule_with_warmup(opt2, num_warmup_steps=20, num_training_steps=STAGE2_EPOCHS * len(dl2))

    train_stage(model, dl2, opt2, sched2, STAGE2_EPOCHS, "stage2")
else:
    print("No construction-specific pairs found — skipping stage 2.")

## 11. Quick validation — sample a few predictions

In [ ]:
model.eval()
for sample in random.sample(val_pairs, min(5, len(val_pairs))):
    image = Image.open(sample["image_path"]).convert("RGB")
    prompt = f"Question: {sample['question']}\nAnswer:"
    inputs = processor(images=image, text=prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=64, do_sample=False)
    pred = processor.tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    print(f"Q: {sample['question']}")
    print(f"GT:   {sample['answer']}")
    print(f"Pred: {pred}")
    print()

## 12. Save checkpoint and push to HuggingFace

In [ ]:
from huggingface_hub import create_repo

checkpoint_dir = Path("/content/checkpoints/spatialvlm-construction")
checkpoint_dir.mkdir(parents=True, exist_ok=True)

model.save_pretrained(checkpoint_dir)
processor.save_pretrained(checkpoint_dir)
print(f"Saved locally to {checkpoint_dir}")

# Push to HuggingFace
create_repo(MODEL_OUT_REPO, repo_type="model", token=HF_TOKEN, private=True, exist_ok=True)
model.push_to_hub(MODEL_OUT_REPO, token=HF_TOKEN)
processor.push_to_hub(MODEL_OUT_REPO, token=HF_TOKEN)
print(f"\nCheckpoint pushed to: https://huggingface.co/{MODEL_OUT_REPO}")

After running, the fine-tuned model is now at `huggingface.co/elizqiu/spatialvlm-construction`.

In the local repo, set:
```
VISION_MODEL_PATH=elizqiu/spatialvlm-construction
```
in `.env` and restart the API server — it will load the fine-tuned weights.